In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("E:/Project/Customer Segmentation/bank_transactions.csv")

In [3]:
df.shape

(1048567, 9)

In [4]:
df.head()

,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
0,T1,C5841053,10-01-1994,F,JAMSHEDPUR,17819.05,02-08-2016,143207,25.0
1,T2,C2142763,04-04-1957,M,JHAJJAR,2270.69,02-08-2016,141858,27999.0
2,T3,C4417068,26-11-1996,F,MUMBAI,17874.44,02-08-2016,142712,459.0
3,T4,C5342380,14-09-1973,F,MUMBAI,866503.21,02-08-2016,142714,2060.0
4,T5,C9031234,24-03-1988,F,NAVI MUMBAI,6714.43,02-08-2016,181156,1762.5


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048567 entries, 0 to 1048566
Data columns (total 9 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   TransactionID            1048567 non-null  object 
 1   CustomerID               1048567 non-null  object 
 2   CustomerDOB              1045170 non-null  object 
 3   CustGender               1047467 non-null  object 
 4   CustLocation             1048416 non-null  object 
 5   CustAccountBalance       1046198 non-null  float64
 6   TransactionDate          1048567 non-null  object 
 7   TransactionTime          1048567 non-null  int64  
 8   TransactionAmount (INR)  1048567 non-null  float64
dtypes: float64(2), int64(1), object(6)
memory usage: 72.0+ MB


In [6]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], dayfirst=True, errors='coerce')

In [7]:
df.dropna(subset=['TransactionDate'], inplace=True)

In [8]:
df.rename(columns={'TransactionAmount (INR)':'Monetary'}, inplace=True)

In [9]:
df.columns= df.columns.str.lower()
df.columns= df.columns.str.replace(' ', '_')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048567 entries, 0 to 1048566
Data columns (total 9 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   transactionid       1048567 non-null  object        
 1   customerid          1048567 non-null  object        
 2   customerdob         1045170 non-null  object        
 3   custgender          1047467 non-null  object        
 4   custlocation        1048416 non-null  object        
 5   custaccountbalance  1046198 non-null  float64       
 6   transactiondate     1048567 non-null  datetime64[ns]
 7   transactiontime     1048567 non-null  int64         
 8   monetary            1048567 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 72.0+ MB


In [11]:
df.to_csv("E:/Project/Customer Segmentation/bank_transactions_clean.csv", index=False)

In [12]:
#RFM Calculation
snapshot_date= df['transactiondate'].max() + pd.Timedelta(days=1)

In [13]:
rfm_df= df.groupby('customerid').agg(
    recency=('transactiondate', lambda x: (snapshot_date - x.max()).days),
    frequency=('transactionid', 'count'),
    monetary=('monetary', 'sum')).reset_index()

In [14]:
print(rfm_df.head())

  customerid  recency  frequency  monetary
0   C1010011       26          2    5106.0
1   C1010012       69          1    1499.0
2   C1010014       76          2    1455.0
3   C1010018       37          1      30.0
4   C1010024       65          1    5000.0


In [15]:
rfm_df.to_csv("E:/Project/Customer Segmentation/rfm_data.csv", index=False)

In [16]:
#RFM Scoring 
rfm_df['R_index']= pd.qcut(rfm_df['recency'], q=5, labels=False, duplicates='drop')
rfm_df['F_index']= pd.qcut(rfm_df['frequency'], q=5, labels=False, duplicates='drop')
rfm_df['M_index']= pd.qcut(rfm_df['monetary'], q=5, labels=False, duplicates='drop')

In [17]:
print(rfm_df.head())

  customerid  recency  frequency  monetary  R_index  F_index  M_index
0   C1010011       26          2    5106.0        0        0        4
1   C1010012       69          1    1499.0        3        0        3
2   C1010014       76          2    1455.0        4        0        3
3   C1010018       37          1      30.0        0        0        0
4   C1010024       65          1    5000.0        3        0        4


In [18]:
rfm_df['R_score']= rfm_df['R_index']+1
rfm_df['F_score']= rfm_df['F_index']+1
rfm_df['M_score']= rfm_df['M_index']+1

In [19]:
max_r_score = rfm_df['R_score'].max()
rfm_df['R_score'] = max_r_score - rfm_df['R_index']

In [20]:
rfm_df['RFM_score'] = rfm_df['R_score'].astype(str) + rfm_df['F_score'].astype(str) + rfm_df['M_score'].astype(str)

In [21]:
print("RFM DataFrame with Scores:")
print(rfm_df[['customerid', 'recency', 'R_score', 'frequency', 'F_score', 'monetary', 'M_score', 'RFM_score']].head())

RFM DataFrame with Scores:
  customerid  recency  R_score  frequency  F_score  monetary  M_score  \
0   C1010011       26        5          2        1    5106.0        5   
1   C1010012       69        2          1        1    1499.0        4   
2   C1010014       76        1          2        1    1455.0        4   
3   C1010018       37        5          1        1      30.0        1   
4   C1010024       65        2          1        1    5000.0        5   

  RFM_score  
0       515  
1       214  
2       114  
3       511  
4       215  


In [22]:
#RFM Segmentation
def rfm_segment(row):
    R = row['R_score']
    F = row['F_score']
    M = row['M_score']

    if R >= 4 and M >= 4:
        return 'Champions' # Bought recently, buy often, spend the most
    elif R >= 4 and M >= 3:
        return 'Loyal Customers' # Good R, F, M
    elif R >= 4 and M < 3:
        return 'New Customers' # High R, Low F and M 
    elif R >= 3:
        return 'Promising' # Average R, Low F and M 
    elif R <= 2 and M >= 4:
        return 'Can\'t Lose Them' # Low R, but high F and M 
    else:
        return 'At Risk/Hibernating' # The rest

In [23]:
rfm_df['segment']= rfm_df.apply(rfm_segment, axis=1)

In [24]:
print("\nSegment Counts:")
print(rfm_df['segment'].value_counts())


Segment Counts:
segment
At Risk/Hibernating    210349
Promising              177585
Champions              158144
New Customers          137962
Can't Lose Them        127013
Loyal Customers         73212
Name: count, dtype: int64


In [25]:
final_rfm_df= rfm_df[['customerid', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']]
final_rfm_df.to_csv('E:/Project/Customer Segmentation/final_rfm_segments.csv', index=False)

In [26]:
df= pd.read_csv("E:/Project/Customer Segmentation/final_rfm_segments.csv")

In [27]:
pip install mysql-connector-python sqlalchemy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [29]:
from sqlalchemy import create_engine

# 1. Set up Database Connection Credentials
username = 'root'
password = '123456'  
host = 'localhost'
port = '3306'
database = 'rfm_project'

# 2. Create the SQLAlchemy Engine
engine = create_engine(f'mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}')

# 3. Load CSV files into Pandas DataFrames
rfm_df = pd.read_csv('E:/Project/Customer Segmentation/final_rfm_segments.csv')
trans_df = pd.read_csv('E:/Project/Customer Segmentation/bank_transactions_clean.csv')

# 4. Define Table Names and Upload
try:
    # Loading RFM Segments
    rfm_df.to_sql(name='customer_segments', con=engine, if_exists='replace', index=False)
    print("Success: 'customer_segments' table created.")

    # Loading Bank Transactions
    trans_df.to_sql(name='bank_transactions', con=engine, if_exists='replace', index=False, chunksize=10000)
    print("Success: 'bank_transactions' table created.")

except Exception as e:
    print(f"An error occurred: {e}")

Success: 'customer_segments' table created.
Success: 'bank_transactions' table created.


# Thank you!